In [ ]:
import json
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import polars as pl
from Bio import SeqIO
from ete4 import Tree

In [ ]:
CWD = Path().resolve()
BASE = CWD.parent

print(f"Current working directory: {CWD}")
print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups_map = {
    "plants_plt": ["solanaceae"],
}

RUN_NAME = "cds_refined"  # output goes to <subgroup>/gene_pair_trees/<RUN_NAME>/
IQTREE_RUN_NAME = "first_pass"  # tree from <subgroup>/cds_tree/iqtree/<IQTREE_RUN_NAME>/tree.nwk

# Every branch length is multiplied by SCALE. The CDS tree is in CDS substitutions/site;
# IGRs evolve faster, so SCALE will later be set from IGR data (IGR/CDS rate ratio).
SCALE = 14.632
SCALE_SOURCE = "igr_trees/cds_first_pass/scale/clustalo_2026-10-08 (weighted median)"

# Branch length for sister tips added when one genome has several IGRs in a gene pair.
DUP_BRANCH = 1e-4

OVERWRITE = False  # True allows changing the settings of an existing RUN_NAME

In [ ]:
def check_run_settings(run_json: Path, settings: dict) -> None:
    """Stop if RUN_NAME already exists with different settings (unless OVERWRITE)."""
    if not run_json.exists():
        return
    old = json.loads(run_json.read_text())
    changed = {k: (old.get(k), v) for k, v in settings.items() if old.get(k) != v}
    if changed and not OVERWRITE:
        details = "; ".join(f"{k}: {a!r} -> {b!r}" for k, (a, b) in changed.items())
        raise RuntimeError(
            f"{run_json.parent.name} already exists with different settings ({details}). "
            "Use a new RUN_NAME, or set OVERWRITE = True."
        )


def scale_tree(tree: Tree, factor: float) -> None:
    """Multiply every branch length in place."""
    if factor == 1:
        return
    for node in tree.traverse():
        if node.dist:
            node.dist *= factor


def gene_pair_tree(subg_tree: Tree, ids_by_an: dict[str, list[str]]) -> Tree:
    """Subgroup tree pruned to the ANs present, tips relabelled to IGR IDs."""
    tree = subg_tree.copy()
    tree.prune(list(ids_by_an), preserve_branch_length=True)

    for leaf in list(tree.leaves()):
        ids = ids_by_an[leaf.name]
        if len(ids) == 1:
            leaf.name = ids[0]
            continue
        # several IGRs from one genome (e.g. IR copies): turn the tip into a node with one child per IGR
        leaf.name = ""
        for igr_id in ids:
            leaf.add_child(name=igr_id, dist=DUP_BRANCH)

    tree.resolve_polytomy()  # 3+ copies from one genome would be a polytomy
    return tree

In [ ]:
for g, subgs in groups_map.items():
    for subg in subgs:
        subg_dir = BASE / g / subg
        print(f"Processing {g}/{subg} | run: {RUN_NAME} | tree: iqtree/{IQTREE_RUN_NAME}")

        gp_file = subg_dir / "gene_pairs_filtered.tsv"
        if not gp_file.exists():
            print(f"  no {gp_file.name}, skipping")
            continue

        iqtree_dir = subg_dir / "cds_tree" / "iqtree" / IQTREE_RUN_NAME
        src_tree_file = iqtree_dir / "tree.nwk"
        iqtree_json = iqtree_dir / "run.json"
        if not src_tree_file.exists():
            raise FileNotFoundError(f"{src_tree_file} not found; run 04_iq_tree with RUN_NAME = {IQTREE_RUN_NAME!r}")
        iqtree_info = json.loads(iqtree_json.read_text()) if iqtree_json.exists() else {}
        if iqtree_info.get("status") != "complete":
            raise RuntimeError(f"iqtree/{IQTREE_RUN_NAME} is not complete (status: {iqtree_info.get('status')})")

        gp_dir = subg_dir / "igr_fastas"
        run_dir = subg_dir / "igr_trees" / RUN_NAME
        run_dir.mkdir(parents=True, exist_ok=True)
        run_json = run_dir / "run.json"

        settings = {"iqtree_run_name": IQTREE_RUN_NAME, "scale": SCALE, "scale_source": SCALE_SOURCE, "dup_branch": DUP_BRANCH}
        check_run_settings(run_json, settings)

        # --- subgroup tree: AN tips, rooted, scaled
        subg_tree = Tree(src_tree_file.read_text(), parser=5)
        scale_tree(subg_tree, SCALE)
        tree_ans = {leaf.name for leaf in subg_tree.leaves()}
        depth = max(subg_tree.get_distance(subg_tree, leaf) for leaf in subg_tree.leaves())
        print(f"  subgroup tree: {len(tree_ans)} tips | max root-to-tip {depth:.4f} (SCALE = {SCALE})")

        # --- IGR ID -> AN
        atlas = pl.read_csv(subg_dir / f"{subg}_summary_atlas.tsv", separator="\t")
        id_to_an = dict(atlas.select(pl.col("ID").cast(pl.String), pl.col("AN").cast(pl.String)).iter_rows())

        # --- pass 1: check every gene pair; collect all problems before writing anything
        gp_df = pl.read_csv(gp_file, separator="\t")
        plans, problems = {}, []
        unplaced_ans: dict[str, int] = defaultdict(int)  # AN not in tree -> number of gene pairs affected
        unknown_ids = 0
        for gp in gp_df["gene_pair"]:
            fasta = gp_dir / f"gp_{gp}.fasta"
            if not fasta.exists():
                problems.append(f"{gp}: missing {fasta.name}")
                continue

            ids_by_an: dict[str, list[str]] = defaultdict(list)
            unplaced = []
            for rec in SeqIO.parse(fasta, "fasta"):
                an = id_to_an.get(rec.id)
                if an in tree_ans:
                    ids_by_an[an].append(rec.id)
                else:
                    unplaced.append(rec.id)
                    if an is None:
                        unknown_ids += 1
                    else:
                        unplaced_ans[an] += 1

            if unplaced:
                # PRANK fails if a FASTA sequence has no tip in the tree
                problems.append(f"{gp}: {len(unplaced)} IGRs without a tip ({unplaced[:3]}...)")
                continue
            if len(ids_by_an) < 2:
                problems.append(f"{gp}: fewer than 2 genomes")
                continue
            plans[gp] = ids_by_an

        if problems:
            print(f"  {len(problems)} of {gp_df.height} gene pairs have problems:")
            for p in problems:
                print(f"    {p}")
            if unplaced_ans:
                print(f"  ANs not in the tree, with the number of gene pairs they appear in:")
                for an, n in sorted(unplaced_ans.items(), key=lambda x: -x[1]):
                    print(f"    {an}: {n}")
            if unknown_ids:
                print(f"  IGR IDs not found in {subg}_summary_atlas.tsv: {unknown_ids}")
            raise RuntimeError(
                f"{g}/{subg}: gene pairs do not match the tree; nothing was written. "
                "Check that 02_gene_pairs was rerun with the current AN list."
            )

        # --- pass 2: write one tree per gene pair
        n_dup_genomes = 0
        for gp, ids_by_an in plans.items():
            n_dup_genomes += sum(len(ids) > 1 for ids in ids_by_an.values())
            gene_pair_tree(subg_tree, ids_by_an).write(outfile=str(run_dir / f"gp_{gp}.nwk"), parser=5)

        stale = {p.stem.removeprefix("gp_") for p in run_dir.glob("gp_*.nwk")} - {str(gp) for gp in plans}

        run_info = {
            "run_name": RUN_NAME,
            "group": g,
            "subgroup": subg,
            "date": f"{datetime.now():%Y-%m-%d %H:%M:%S}",
            **settings,
            "source_tree": str(src_tree_file.relative_to(subg_dir)),
            "source_tree_abs": str(src_tree_file.resolve()),
            "mafft_run_name": iqtree_info.get("mafft_run_name"),
            "gene_pairs_file": str(gp_file.relative_to(subg_dir)),
            "gene_pairs_file_abs": str(gp_file.resolve()),
            "gene_pair_fastas": str(gp_dir.relative_to(subg_dir)),
            "gene_pair_fastas_abs": str(gp_dir.resolve()),
            "n_tree_tips": len(tree_ans),
            "max_root_to_tip": depth,
            "n_gene_pair_trees": len(plans),
        }
        run_json.write_text(json.dumps(run_info, indent=2))

        print(
            f"  gene-pair trees: {len(plans)}/{gp_df.height} written -> {run_dir.relative_to(subg_dir)}/"
            f" | genomes with several IGRs (sister tips): {n_dup_genomes}"
        )
        if stale:
            print(f"  note: {len(stale)} trees in {run_dir.relative_to(subg_dir)}/ are for gene pairs no longer listed")

print(f"Done with all groups/subgroups for run {RUN_NAME}")